# ⚙️ Python Methods — Deep Dive & Advanced Patterns

> **Module:** Object-Oriented Programming (OOP) | **Notebook 2 of 5**

In Python OOP, **methods** define the behaviors, transformations, and communication contracts of objects. While functions exist independently, methods are bound to classes or instances through Python's **descriptor protocol**.

This notebook explores the complete spectrum of methods in Python: from method binding mechanics and fluent method chaining to polymorphic dispatch (`@singledispatchmethod`), abstract interfaces (`@abstractmethod`), method decorators, and essential dunder (magic) methods.

---

## 📋 Table of Contents
1. [The Anatomy of a Method & The Descriptor Binding Protocol](#1.-The-Anatomy-of-a-Method-&-The-Descriptor-Binding-Protocol)
2. [Instance Methods & Method Chaining (Fluent Interfaces)](#2.-Instance-Methods-&-Method-Chaining-(Fluent-Interfaces))
3. [Class Methods (`@classmethod`) & Polymorphic Subclass Factories](#3.-Class-Methods-(@classmethod)-&-Polymorphic-Subclass-Factories)
4. [Static Methods (`@staticmethod`) & Utility Namespacing](#4.-Static-Methods-(@staticmethod)-&-Utility-Namespacing)
5. [Method Overloading & Polymorphic Single-Dispatch](#5.-Method-Overloading-&-Polymorphic-Single-Dispatch)
6. [Abstract Methods & Interface Contracts (`abc.ABC`)](#6.-Abstract-Methods-&-Interface-Contracts-(abc.ABC))
7. [Essential Dunder (Magic) Methods for Custom Behaviors](#7.-Essential-Dunder-(Magic)-Methods-for-Custom-Behaviors)
8. [Method Decorators & Behavioral Interceptors](#8.-Method-Decorators-&-Behavioral-Interceptors)
9. [Real-World Case Studies & Architectural Patterns](#9.-Real-World-Case-Studies-&-Architectural-Patterns)
   - 9.1 [Fluent SQL Query & Pipeline Builder](#9.1-Fluent-SQL-Query-&-Pipeline-Builder)
   - 9.2 [Polymorphic Event Ingestion Pipeline](#9.2-Polymorphic-Event-Ingestion-Pipeline)
   - 9.3 [Financial Currency & Arithmetic Engine](#9.3-Financial-Currency-&-Arithmetic-Engine)
10. [Common Pitfalls & Anti-Patterns](#10.-Common-Pitfalls-&-Anti-Patterns)
11. [Hands-On Interactive Challenges](#11.-Hands-On-Interactive-Challenges)
12. [Quick Reference Card & Summary](#12.-Quick-Reference-Card-&-Summary)


---
## 1. The Anatomy of a Method & The Descriptor Binding Protocol

### 🔍 Function vs. Method: What Really Happens?
When you define a function inside a `class` body, Python creates a standard **function object**. 
When that function is accessed via an **instance**, Python's **descriptor protocol** converts it into a **bound method object**.

```text
Class definition:
   class Greeter:
       def greet(self): ...

1. Access via Class:     Greeter.greet           --> <function Greeter.greet> (Unbound Function)
2. Access via Instance:  obj = Greeter()
                         obj.greet               --> <bound method Greeter.greet of <Greeter obj>>
3. Calling obj.greet():  Translates to -->       Greeter.greet.__get__(obj, Greeter)()
```


In [ ]:
class Greeter:
    """Demonstrates method binding mechanics."""
    
    def greet(self, name: str) -> str:
        return f"Hello, {name}!"

# 1. Accessing through the Class: Raw function
class_attr = Greeter.greet
print(f"Greeter.greet type: {type(class_attr)}")
print(f"Is function? {hasattr(class_attr, '__code__')}")

# 2. Accessing through an Instance: Bound method
g = Greeter()
instance_attr = g.greet
print(f"\ng.greet type: {type(instance_attr)}")
print(f"Bound to instance (__self__): {instance_attr.__self__}")
print(f"Underlying raw function (__func__): {instance_attr.__func__}")
print(f"Qualified name (__qualname__): {instance_attr.__qualname__}")

# 3. Manual Descriptor Binding Proof:
# Calling the descriptor's __get__ method produces the exact same bound method:
bound_manual = Greeter.greet.__get__(g, Greeter)
print(f"\nManual bound call: {bound_manual('Developer')}")
print(f"Standard method call: {g.greet('Developer')}")
print(f"Class function call:  {Greeter.greet(g, 'Developer')}")


---
## 2. Instance Methods & Method Chaining (Fluent Interfaces)

### 🔗 What is Method Chaining (Fluent Interface)?
Method chaining is an object-oriented design pattern where multiple methods are called sequentially on a single object in a single line of code (`obj.step1().step2().step3()`).

To enable chaining, each mutating method **returns `self`** instead of `None`.


In [ ]:
class StringBuilder:
    """A fluent string builder demonstrating method chaining."""
    
    def __init__(self, initial_text: str = ""):
        self._parts: list[str] = [initial_text] if initial_text else []

    def append(self, text: str) -> 'StringBuilder':
        """Appends text and returns self for chaining."""
        self._parts.append(text)
        return self

    def append_line(self, text: str = "") -> 'StringBuilder':
        """Appends text followed by newline and returns self."""
        self._parts.append(text + "\n")
        return self

    def wrap_with(self, prefix: str, suffix: str) -> 'StringBuilder':
        """Wraps the entire accumulated content."""
        content = "".join(self._parts)
        self._parts = [prefix, content, suffix]
        return self

    def build(self) -> str:
        """Terminal method that produces the final formatted string."""
        return "".join(self._parts)

# Fluent chaining in action:
builder = StringBuilder()
result = (
    builder
    .append("SELECT id, name, email")
    .append(" FROM users")
    .append(" WHERE active = 1")
    .wrap_with("/* QUERY BEGIN */ ", " /* QUERY END */")
    .build()
)

print("Generated SQL Query:")
print(result)


---
## 3. Class Methods (`@classmethod`) & Polymorphic Subclass Factories

### 🏭 Why `@classmethod` is Essential for Inheritance
A `@classmethod` receives the class itself (`cls`) as its first argument.

When used as an **alternative constructor** (Factory Pattern), `cls(...)` ensures that subclasses instantiate **the correct subclass type**, rather than being hardcoded to the base class!


In [ ]:
import json
from datetime import datetime

class BaseDocument:
    """Base document with subclass-aware factory constructors."""
    
    def __init__(self, title: str, author: str, content: str):
        self.title = title
        self.author = author
        self.content = content
        self.created_at = datetime.now()

    @classmethod
    def from_json(cls, json_str: str) -> 'BaseDocument':
        """Subclass-aware factory constructor: uses cls(...) instead of BaseDocument(...)"""
        data = json.loads(json_str)
        return cls(
            title=data["title"],
            author=data["author"],
            content=data["content"]
        )

    @classmethod
    def from_dict(cls, data: dict) -> 'BaseDocument':
        return cls(
            title=data["title"],
            author=data["author"],
            content=data["content"]
        )

    def summary(self) -> str:
        return f"[{self.__class__.__name__}] '{self.title}' by {self.author} ({len(self.content)} chars)"

class TechnicalReport(BaseDocument):
    """Subclass inheriting from BaseDocument."""
    pass

class LegalContract(BaseDocument):
    """Subclass inheriting from BaseDocument."""
    pass

# Demonstrate polymorphic factory instantiation
json_payload = '{"title": "Q3 Architecture Review", "author": "Alice", "content": "Microservices migration..."}'

doc1 = BaseDocument.from_json(json_payload)
doc2 = TechnicalReport.from_json(json_payload)
doc3 = LegalContract.from_json(json_payload)

print(f"doc1 instance type: {type(doc1).__name__} -> {doc1.summary()}")
print(f"doc2 instance type: {type(doc2).__name__} -> {doc2.summary()}")
print(f"doc3 instance type: {type(doc3).__name__} -> {doc3.summary()}")

assert isinstance(doc2, TechnicalReport)
assert isinstance(doc3, LegalContract)
print("\n[OK] Subclass-aware factory correctly instantiated derived types!")


---
## 4. Static Methods (`@staticmethod`) & Utility Namespacing

### 🧭 Method Types Comparison & When to Use Which

| Question | Use Instance Method (`self`) | Use Class Method (`@classmethod`, `cls`) | Use Static Method (`@staticmethod`) |
| :--- | :--- | :--- | :--- |
| Does the method need to read or mutate instance state? | **Yes** | No | No |
| Does the method act as an alternative constructor? | No | **Yes** | No |
| Does the method modify class-level state / registry? | No | **Yes** | No |
| Is the method an isolated utility function related to the class? | No | No | **Yes** |


In [ ]:
import re

class EmailValidator:
    """Demonstrates static utility isolation within a logical namespace."""
    
    EMAIL_REGEX = re.compile(r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$")

    @staticmethod
    def is_valid_email(email: str) -> bool:
        """Validates email syntax without requiring instance or class state."""
        if not isinstance(email, str):
            return False
        return bool(EmailValidator.EMAIL_REGEX.match(email.strip()))

    @staticmethod
    def extract_domain(email: str) -> str | None:
        """Extracts domain portion from a valid email."""
        if not EmailValidator.is_valid_email(email):
            return None
        return email.split("@")[1].lower().strip()

    @staticmethod
    def mask_email(email: str) -> str:
        """Masks user portion for privacy compliance (e.g. j***@example.com)."""
        if not EmailValidator.is_valid_email(email):
            return "INVALID_EMAIL"
        user, domain = email.split("@")
        if len(user) <= 2:
            masked_user = user[0] + "*"
        else:
            masked_user = user[0] + "*" * (len(user) - 2) + user[-1]
        return f"{masked_user}@{domain}"

# Testing static utilities
test_emails = ["alice@example.com", "bob.smith+tag@company.org", "invalid-email@", "charlie@gov.us"]

for email in test_emails:
    valid = EmailValidator.is_valid_email(email)
    domain = EmailValidator.extract_domain(email)
    masked = EmailValidator.mask_email(email)
    print(f"Email: {email:<28} | Valid: {str(valid):<5} | Domain: {str(domain):<12} | Masked: {masked}")


---
## 5. Method Overloading & Polymorphic Single-Dispatch

### ❓ Why Doesn't Python Support Traditional Method Overloading?
In statically typed languages (Java, C++), multiple methods can share the same name if they have different parameter types. In Python, methods are bound by name in the class dictionary — defining a method twice simply **overwrites** the first definition.

### 🛠️ The Python Solutions:
1. **Dynamic Type Checking** (`isinstance`, default arguments)
2. **`@functools.singledispatchmethod`** (Modern, extensible, clean single-dispatch polymorphism)
3. **`@typing.overload`** (For static type checkers like Mypy/Pyright)


In [ ]:
from functools import singledispatchmethod

class DataFormatter:
    """Polymorphic data formatter using singledispatchmethod."""

    # Default fallback handler for unsupported types
    @singledispatchmethod
    def format_data(self, data) -> str:
        raise NotImplementedError(f"Cannot format unsupported data type: {type(data).__name__}")

    # Specialized handler for integers
    @format_data.register
    def _(self, data: int) -> str:
        return f"[INTEGER] {data:,d} (Hex: {hex(data)})"

    # Specialized handler for floats
    @format_data.register
    def _(self, data: float) -> str:
        return f"[FLOAT] {data:,.4f} (Scientific: {data:.2e})"

    # Specialized handler for strings
    @format_data.register
    def _(self, data: str) -> str:
        return f"[STRING] '{data.strip()}' (Length: {len(data)})"

    # Specialized handler for lists
    @format_data.register
    def _(self, data: list) -> str:
        formatted_items = ", ".join(repr(x) for x in data[:3])
        suffix = "..." if len(data) > 3 else ""
        return f"[LIST] [{formatted_items}{suffix}] (Total elements: {len(data)})"

    # Specialized handler for dictionaries
    @format_data.register
    def _(self, data: dict) -> str:
        keys_summary = ", ".join(str(k) for k in list(data.keys())[:3])
        return f"[DICT] Keys: {{{keys_summary}}} (Size: {len(data)})"

formatter = DataFormatter()

print(formatter.format_data(1000000))
print(formatter.format_data(3.14159265))
print(formatter.format_data("  Python OOP Methods  "))
print(formatter.format_data([10, 20, 30, 40, 50]))
print(formatter.format_data({"id": 101, "name": "Prod", "active": True}))

# Unsupported type error demonstration
try:
    formatter.format_data(object())
except NotImplementedError as e:
    print(f"\nCaught expected error: {e}")


---
## 6. Abstract Methods & Interface Contracts (`abc.ABC`)

### 📜 What is an Abstract Method?
An abstract method is a method declared in a base class that **must be implemented** by all concrete subclasses. If a subclass fails to implement any abstract method, attempting to instantiate it raises a `TypeError`.

Python provides the `abc` (Abstract Base Classes) module for defining formal interface contracts.


In [ ]:
from abc import ABC, abstractmethod

class PaymentGateway(ABC):
    """Abstract base class defining the payment gateway contract."""

    @abstractmethod
    def authenticate(self, api_key: str) -> bool:
        """Must authenticate credentials."""
        pass

    @abstractmethod
    def process_payment(self, amount: float, currency: str) -> dict:
        """Must charge the given amount and return a transaction receipt."""
        pass

    @abstractmethod
    def refund(self, transaction_id: str, amount: float) -> bool:
        """Must refund a previously processed transaction."""
        pass

    # Concrete helper method shared by all implementations
    def format_currency(self, amount: float, currency: str) -> str:
        return f"{currency.upper()} {amount:,.2f}"

class StripeGateway(PaymentGateway):
    def __init__(self, api_key: str):
        self.api_key = api_key
        self._authenticated = False

    def authenticate(self, api_key: str) -> bool:
        self._authenticated = (api_key.startswith("sk_live_"))
        return self._authenticated

    def process_payment(self, amount: float, currency: str) -> dict:
        if not self._authenticated:
            raise PermissionError("Stripe gateway not authenticated.")
        return {
            "status": "SUCCESS",
            "provider": "Stripe",
            "amount_charged": self.format_currency(amount, currency),
            "tx_id": "ch_3N4xYz2eZvKYlo2C"
        }

    def refund(self, transaction_id: str, amount: float) -> bool:
        return True

# 1. Attempting to instantiate the abstract class raises TypeError:
try:
    gw = PaymentGateway()
except TypeError as e:
    print(f"Cannot instantiate ABC directly: {e}")

# 2. Concrete implementation works as expected:
stripe = StripeGateway("sk_live_987654321")
stripe.authenticate("sk_live_987654321")
receipt = stripe.process_payment(249.99, "usd")
print(f"\nStripe payment receipt: {receipt}")


---
## 7. Essential Dunder (Magic) Methods for Custom Behaviors

Dunder (double underscore) methods allow custom classes to integrate directly with Python's built-in syntax and standard protocols.

| Dunder Method | Python Syntax / Trigger | Purpose |
| :--- | :--- | :--- |
| `__repr__(self)` | `repr(obj)`, interactive shell | Official, unambiguous string representation (for developers) |
| `__str__(self)` | `str(obj)`, `print(obj)` | Readable, user-facing string representation |
| `__call__(self, ...)` | `obj(*args, **kwargs)` | Makes the instance callable like a function |
| `__eq__(self, other)` | `obj1 == obj2` | Equality comparison |
| `__lt__(self, other)` | `obj1 < obj2`, `sorted()` | Less-than comparison (enables sorting) |
| `__add__(self, other)` | `obj1 + obj2` | Binary addition operator |
| `__len__(self)` | `len(obj)` | Returns length / size |
| `__getitem__(self, k)`| `obj[k]` | Indexing / subscription access |
| `__contains__(self, item)` | `item in obj` | Membership test |


In [ ]:
from functools import total_ordering

@total_ordering  # Automatically provides __le__, __gt__, __ge__ from __eq__ and __lt__
class PriorityTask:
    """Demonstrates dunder methods: representation, comparisons, and calling."""

    def __init__(self, title: str, priority: int, duration_mins: int):
        self.title = title
        self.priority = priority        # Lower number = higher priority
        self.duration_mins = duration_mins

    # 1. Official representation
    def __repr__(self) -> str:
        return f"PriorityTask(title='{self.title}', priority={self.priority}, duration_mins={self.duration_mins})"

    # 2. User-friendly string
    def __str__(self) -> str:
        return f"[P{self.priority}] {self.title} ({self.duration_mins}m)"

    # 3. Rich comparison for sorting (lowest priority number first)
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, PriorityTask):
            return NotImplemented
        return (self.priority, self.duration_mins) == (other.priority, other.duration_mins)

    def __lt__(self, other: object) -> bool:
        if not isinstance(other, PriorityTask):
            return NotImplemented
        return (self.priority, self.duration_mins) < (other.priority, other.duration_mins)

    # 4. Callable object protocol: executing task directly
    def __call__(self, executor: str = "System") -> str:
        return f"Executing task '{self.title}' by {executor}... Done!"

# Create tasks
t1 = PriorityTask("Fix Critical Security Bug", priority=1, duration_mins=45)
t2 = PriorityTask("Update Documentation", priority=3, duration_mins=30)
t3 = PriorityTask("Deploy to Staging", priority=2, duration_mins=15)

tasks = [t2, t1, t3]

print("String representations:")
print(f"str(t1):  {str(t1)}")
print(f"repr(t1): {repr(t1)}")

print("\nSorted tasks by priority:")
for task in sorted(tasks):
    print(f"  -> {task}")

print(f"\nIs t1 < t2? {t1 < t2} (P1 has higher precedence than P3)")

# Calling task instance directly via __call__:
print(f"Callable execution: {t1('DevOps-Agent')}")


---
## 8. Method Decorators & Behavioral Interceptors

### 🛡️ Intercepting Method Calls
Decorators wrap method execution to add cross-cutting concerns like:
- Execution timing & profiling
- Input argument validation
- Authentication & role checking
- Retry logic on failure


In [ ]:
import time
import functools

def log_execution_time(func):
    """Decorator that measures and logs execution duration of a method."""
    @functools.wraps(func)
    def wrapper(self, *args, **kwargs):
        start = time.perf_counter()
        result = func(self, *args, **kwargs)
        duration_ms = (time.perf_counter() - start) * 1000
        print(f"[METRICS] {self.__class__.__name__}.{func.__name__} took {duration_ms:.3f} ms")
        return result
    return wrapper

def require_positive_args(func):
    """Decorator that validates all positional numeric arguments are strictly positive."""
    @functools.wraps(func)
    def wrapper(self, *args, **kwargs):
        for arg in args:
            if isinstance(arg, (int, float)) and arg <= 0:
                raise ValueError(f"Method '{func.__name__}' requires strictly positive arguments; received {arg}")
        return func(self, *args, **kwargs)
    return wrapper

class Calculator:
    @log_execution_time
    @require_positive_args
    def compute_compound_growth(self, principal: float, rate: float, periods: int) -> float:
        total = principal
        for _ in range(periods):
            total *= (1 + rate)
        return total

calc = Calculator()
growth = calc.compute_compound_growth(10000.0, 0.07, 10)
print(f"10-year growth: ${growth:,.2f}")

try:
    calc.compute_compound_growth(-500.0, 0.05, 5)
except ValueError as e:
    print(f"\nCaught expected validation error: {e}")


---
## 9. Real-World Case Studies & Architectural Patterns

---

### 9.1 Fluent SQL Query & Pipeline Builder
A full-featured Query Builder implementing method chaining with parameterized output.


In [ ]:
class QueryBuilder:
    """Fluent API SQL Query Builder."""
    
    def __init__(self, table: str):
        self._table: str = table
        self._fields: list[str] = ["*"]
        self._conditions: list[str] = []
        self._params: list = []
        self._order_by: str | None = None
        self._limit_val: int | None = None

    def select(self, *fields: str) -> 'QueryBuilder':
        if fields:
            self._fields = list(fields)
        return self

    def where(self, condition: str, *params) -> 'QueryBuilder':
        self._conditions.append(condition)
        self._params.extend(params)
        return self

    def order_by(self, field: str, direction: str = "ASC") -> 'QueryBuilder':
        direction_upper = direction.upper().strip()
        if direction_upper not in {"ASC", "DESC"}:
            raise ValueError("Direction must be 'ASC' or 'DESC'")
        self._order_by = f"{field} {direction_upper}"
        return self

    def limit(self, count: int) -> 'QueryBuilder':
        if count <= 0:
            raise ValueError("Limit must be positive")
        self._limit_val = count
        return self

    def build(self) -> tuple[str, tuple]:
        """Constructs the final SQL query and parameter tuple."""
        query_parts = [f"SELECT {', '.join(self._fields)} FROM {self._table}"]
        
        if self._conditions:
            query_parts.append(f"WHERE {' AND '.join(self._conditions)}")
        if self._order_by:
            query_parts.append(f"ORDER BY {self._order_by}")
        if self._limit_val is not None:
            query_parts.append(f"LIMIT {self._limit_val}")
            
        sql = " ".join(query_parts) + ";"
        return sql, tuple(self._params)

# Fluent chaining demonstration
sql, params = (
    QueryBuilder("customers")
    .select("id", "first_name", "last_name", "credit_score")
    .where("active = ?", True)
    .where("credit_score >= ?", 700)
    .where("country = ?", "USA")
    .order_by("credit_score", "DESC")
    .limit(10)
    .build()
)

print(f"Generated SQL: {sql}")
print(f"Bound Params:  {params}")


---
### 9.2 Polymorphic Event Ingestion Pipeline
Processing heterogeneous event objects cleanly with `@singledispatchmethod`.


In [ ]:
from functools import singledispatchmethod

# Domain event classes
class UserSignupEvent:
    def __init__(self, user_id: str, email: str):
        self.user_id = user_id
        self.email = email

class OrderPlacedEvent:
    def __init__(self, order_id: str, amount: float, items_count: int):
        self.order_id = order_id
        self.amount = amount
        self.items_count = items_count

class SystemAlertEvent:
    def __init__(self, severity: str, message: str):
        self.severity = severity
        self.message = message

class EventDispatcher:
    """Dispatches domain events to dedicated handlers polymorphically."""

    def __init__(self):
        self.processed_count: int = 0

    @singledispatchmethod
    def handle_event(self, event) -> str:
        raise ValueError(f"Unknown event type: {type(event).__name__}")

    @handle_event.register
    def _(self, event: UserSignupEvent) -> str:
        self.processed_count += 1
        return f"[SIGNUP] Welcome email queued for {event.email} (ID: {event.user_id})"

    @handle_event.register
    def _(self, event: OrderPlacedEvent) -> str:
        self.processed_count += 1
        return f"[ORDER] Payment processed for Order #{event.order_id}: ${event.amount:,.2f} ({event.items_count} items)"

    @handle_event.register
    def _(self, event: SystemAlertEvent) -> str:
        self.processed_count += 1
        return f"[ALERT:{event.severity.upper()}] Notification broadcast: '{event.message}'"

dispatcher = EventDispatcher()
events = [
    UserSignupEvent("U-9901", "grace.hopper@navy.mil"),
    OrderPlacedEvent("ORD-7741", 1499.50, 3),
    SystemAlertEvent("CRITICAL", "High CPU utilization detected on node-04")
]

for ev in events:
    print(dispatcher.handle_event(ev))

print(f"\nTotal events handled: {dispatcher.processed_count}")


---
### 9.3 Financial Currency & Arithmetic Engine
Demonstrates arithmetic operator overloading (`__add__`, `__sub__`, `__mul__`), equality, and rich formatting.


In [ ]:
from functools import total_ordering

@total_ordering
class Money:
    """Immutable monetary amount with currency conversion support."""
    __slots__ = ('_amount_cents', '_currency')
    
    # Class currency exchange rates relative to USD
    EXCHANGE_RATES: dict[str, float] = {
        "USD": 1.0,
        "EUR": 1.08,
        "GBP": 1.27,
        "JPY": 0.0068
    }

    def __init__(self, amount: float | int, currency: str = "USD"):
        currency = currency.upper().strip()
        if currency not in Money.EXCHANGE_RATES:
            raise ValueError(f"Unsupported currency: {currency}")
        self._amount_cents = round(float(amount) * 100)
        self._currency = currency

    @property
    def amount(self) -> float:
        return self._amount_cents / 100.0

    @property
    def currency(self) -> str:
        return self._currency

    def to_currency(self, target_currency: str) -> 'Money':
        """Converts this Money instance to target currency."""
        target_currency = target_currency.upper().strip()
        if target_currency not in Money.EXCHANGE_RATES:
            raise ValueError(f"Unsupported target currency: {target_currency}")
        usd_value = (self.amount) * Money.EXCHANGE_RATES[self._currency]
        converted_amount = usd_value / Money.EXCHANGE_RATES[target_currency]
        return Money(converted_amount, target_currency)

    # 1. Operator: Addition (+)
    def __add__(self, other: 'Money') -> 'Money':
        if not isinstance(other, Money):
            return NotImplemented
        converted_other = other.to_currency(self._currency)
        return Money(self.amount + converted_other.amount, self._currency)

    # 2. Operator: Subtraction (-)
    def __sub__(self, other: 'Money') -> 'Money':
        if not isinstance(other, Money):
            return NotImplemented
        converted_other = other.to_currency(self._currency)
        return Money(self.amount - converted_other.amount, self._currency)

    # 3. Operator: Scalar Multiplication (*)
    def __mul__(self, factor: float | int) -> 'Money':
        if not isinstance(factor, (int, float)):
            return NotImplemented
        return Money(self.amount * factor, self._currency)

    # 4. Equality & Comparison
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Money):
            return NotImplemented
        converted_other = other.to_currency("USD")
        this_usd = self.to_currency("USD")
        return abs(this_usd.amount - converted_other.amount) < 0.01

    def __lt__(self, other: 'Money') -> bool:
        if not isinstance(other, Money):
            return NotImplemented
        return self.to_currency("USD").amount < other.to_currency("USD").amount

    def __repr__(self) -> str:
        return f"Money({self.amount:.2f}, '{self._currency}')"

    def __str__(self) -> str:
        return f"{self._currency} {self.amount:,.2f}"

# Demonstration:
m1 = Money(100.0, "USD")
m2 = Money(50.0, "EUR")

print(f"m1: {m1}")
print(f"m2: {m2} -> in USD: {m2.to_currency('USD')}")

total = m1 + m2
print(f"m1 + m2 (in USD): {total}")

tripled = m1 * 3
print(f"m1 * 3: {tripled}")

print(f"Is $100 USD > 50 EUR? {m1 > m2}")


---
## 10. Common Pitfalls & Anti-Patterns

### ❌ Pitfall 1: Forgetting `self` in Method Definition
Defining a method without `self` causes Python to pass the instance anyway, resulting in:
`TypeError: my_method() takes 0 positional arguments but 1 was given`

### ❌ Pitfall 2: Naively Caching Instance Methods with `@functools.lru_cache`
Applying `@lru_cache` directly on an instance method creates a hidden reference in the cache pointing to `self`. This prevents the garbage collector from reclaiming the object, leading to **silent memory leaks**! Use caching on static methods or decoupled functions instead.

### ❌ Pitfall 3: Inconsistent Equality (`__eq__`) and Hashing (`__hash__`)
If you define `__eq__` without `__hash__`, Python automatically sets `__hash__ = None`, making instances **unhashable** (they cannot be added to `set` or used as `dict` keys). If an object is immutable and defines `__eq__`, always implement `__hash__`!


---
## 11. Hands-On Interactive Challenges

---

### 🎯 Challenge 1: Fluent HTML Tag Builder
Implement an `HtmlElement` class supporting method chaining:
- Attributes: `tag: str`, `classes: list[str]`, `attributes: dict[str, str]`, `text: str`.
- Chaining methods: `add_class(cls_name: str)`, `set_attribute(key: str, value: str)`, `set_text(text: str)`.
- Terminal method `render() -> str` outputting valid HTML tag (e.g. `<button class="btn btn-primary" id="submit-btn">Submit</button>`).


In [ ]:
class HtmlElement:
    def __init__(self, tag: str):
        self.tag = tag.strip().lower()
        self.classes: list[str] = []
        self.attributes: dict[str, str] = {}
        self.text: str = ""

    def add_class(self, cls_name: str) -> 'HtmlElement':
        cls_name = cls_name.strip()
        if cls_name and cls_name not in self.classes:
            self.classes.append(cls_name)
        return self

    def set_attribute(self, key: str, value: str) -> 'HtmlElement':
        self.attributes[key.strip()] = str(value).strip()
        return self

    def set_text(self, text: str) -> 'HtmlElement':
        self.text = text
        return self

    def render(self) -> str:
        attrs = []
        if self.classes:
            attrs.append(f'class="{" ".join(self.classes)}"')
        for k, v in self.attributes.items():
            attrs.append(f'{k}="{v}"')
            
        attr_str = f" {' '.join(attrs)}" if attrs else ""
        return f"<{self.tag}{attr_str}>{self.text}</{self.tag}>"

# Automated verification test
html = (
    HtmlElement("button")
    .add_class("btn")
    .add_class("btn-primary")
    .set_attribute("id", "submit-btn")
    .set_text("Submit Form")
    .render()
)

assert html == '<button class="btn btn-primary" id="submit-btn">Submit Form</button>'
print("[OK] Challenge 1 Passed! Rendered:", html)


---
### 🎯 Challenge 2: Polymorphic Serializer with `@singledispatchmethod`
Implement a `TypeSerializer` class with `serialize(data)` that:
- For `int` / `float`: returns string formatted number (`"NUM:100"` / `"NUM:3.14"`).
- For `str`: returns uppercase quoted string (`"STR:'HELLO'"`).
- For `list` / `tuple`: returns comma-separated serialized elements wrapped in brackets (`"[NUM:1, NUM:2]"`).
- Raises `TypeError` for unsupported types.


In [ ]:
from functools import singledispatchmethod

class TypeSerializer:
    @singledispatchmethod
    def serialize(self, data) -> str:
        raise TypeError(f"Unsupported serialization type: {type(data).__name__}")

    @serialize.register(int)
    @serialize.register(float)
    def _(self, data: int | float) -> str:
        return f"NUM:{data}"

    @serialize.register(str)
    def _(self, data: str) -> str:
        return f"STR:'{data.upper()}'"

    @serialize.register(list)
    @serialize.register(tuple)
    def _(self, data: list | tuple) -> str:
        inner = ", ".join(self.serialize(x) for x in data)
        return f"[{inner}]"

# Automated verification test
serializer = TypeSerializer()

assert serializer.serialize(42) == "NUM:42"
assert serializer.serialize(3.14) == "NUM:3.14"
assert serializer.serialize("test") == "STR:'TEST'"
assert serializer.serialize([1, "hi", 2.5]) == "[NUM:1, STR:'HI', NUM:2.5]"

try:
    serializer.serialize(set())
    assert False, "Should raise TypeError"
except TypeError:
    pass

print("[OK] Challenge 2 Passed!")


---
### 🎯 Challenge 3: Vector Math with Operator Overloading
Create a `Vector3D` class supporting `+`, `-`, `*` (scalar), `==`, and `abs()` (vector length / magnitude).


In [ ]:
import math

class Vector3D:
    __slots__ = ('x', 'y', 'z')

    def __init__(self, x: float, y: float, z: float):
        self.x = float(x)
        self.y = float(y)
        self.z = float(z)

    def __add__(self, other: 'Vector3D') -> 'Vector3D':
        if not isinstance(other, Vector3D):
            return NotImplemented
        return Vector3D(self.x + other.x, self.y + other.y, self.z + other.z)

    def __sub__(self, other: 'Vector3D') -> 'Vector3D':
        if not isinstance(other, Vector3D):
            return NotImplemented
        return Vector3D(self.x - other.x, self.y - other.y, self.z - other.z)

    def __mul__(self, factor: float | int) -> 'Vector3D':
        if not isinstance(factor, (int, float)):
            return NotImplemented
        return Vector3D(self.x * factor, self.y * factor, self.z * factor)

    def __abs__(self) -> float:
        return math.sqrt(self.x**2 + self.y**2 + self.z**2)

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Vector3D):
            return NotImplemented
        return (self.x, self.y, self.z) == (other.x, other.y, other.z)

    def __repr__(self) -> str:
        return f"Vector3D({self.x:.1f}, {self.y:.1f}, {self.z:.1f})"

# Automated verification test
v1 = Vector3D(1, 2, 3)
v2 = Vector3D(4, 5, 6)

assert (v1 + v2) == Vector3D(5, 7, 9)
assert (v2 - v1) == Vector3D(3, 3, 3)
assert (v1 * 2) == Vector3D(2, 4, 6)
assert abs(Vector3D(3, 0, 4)) == 5.0

print("[OK] Challenge 3 Passed!")


---
## 12. Quick Reference Card & Summary

### 💡 All Method Patterns in One Runnable Cell


In [ ]:
# ============================================================
# PYTHON OOP METHODS — QUICK REFERENCE CARD
# ============================================================

from abc import ABC, abstractmethod
from functools import singledispatchmethod

class QuickRefEntity(ABC):
    """Complete summary of method paradigms in Python."""
    
    def __init__(self, name: str, value: float):
        self.name = name
        self.value = value

    # 1. Instance Method (reads/modifies instance state)
    def update_value(self, delta: float) -> 'QuickRefEntity':
        self.value += delta
        return self  # Return self for fluent chaining

    # 2. Class Method (alternative constructor / class-level state)
    @classmethod
    def from_tuple(cls, data: tuple[str, float]) -> 'QuickRefEntity':
        return cls(data[0], data[1])

    # 3. Static Method (isolated namespace utility)
    @staticmethod
    def is_positive(num: float) -> bool:
        return num > 0

    # 4. Abstract Method (contract enforcement for subclasses)
    @abstractmethod
    def compute_metric(self) -> float:
        pass

    # 5. Dunder Methods (custom representation, operator overloading, callable)
    def __repr__(self) -> str:
        return f"{self.__class__.__name__}(name='{self.name}', value={self.value})"

    def __add__(self, other: 'QuickRefEntity') -> float:
        return self.value + other.value

    def __call__(self) -> str:
        return f"Executing {self.name} -> metric={self.compute_metric()}"

class ConcreteEntity(QuickRefEntity):
    def compute_metric(self) -> float:
        return self.value * 2.0

# Run quick reference validation
item1 = ConcreteEntity("ItemA", 50.0)
item2 = ConcreteEntity.from_tuple(("ItemB", 75.0))

item1.update_value(10.0)  # Method chaining
total_val = item1 + item2  # Operator overloading

print(f"Created: {item1} & {item2}")
print(f"Total value (item1 + item2): {total_val}")
print(f"Callable execution: {item1()}")
print(f"Static method check (is_positive): {QuickRefEntity.is_positive(item1.value)}")


### 📊 Master Method Matrix

| Method Category | Syntax / Decorator | 1st Argument | Primary Purpose |
| :--- | :--- | :--- | :--- |
| **Instance Method** | `def method(self):` | `self` | Manipulating and accessing instance state |
| **Fluent Method** | `return self` | `self` | Enabling chained API pipelines (`.a().b().c()`) |
| **Class Method** | `@classmethod` | `cls` | Subclass-safe factory constructors, class state |
| **Static Method** | `@staticmethod` | *None* | Decoupled utility scoped within class namespace |
| **Single-Dispatch** | `@singledispatchmethod` | `self, arg` | Polymorphic multi-type argument handling |
| **Abstract Method** | `@abstractmethod` | `self` / `cls` | Enforcing strict API contracts in base classes |
| **Dunder Methods** | `__repr__`, `__add__`, etc. | `self` | Native Python syntax & operator overloading |

---
*Next up in OOP Series → **Inheritance & Polymorphism (`inheritance_and_polymorphism.ipynb`)***
